# Import libraries

In [29]:
import os
import cohere
from dotenv import load_dotenv
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.schema import Document
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

True

# Initialize both docs and retriever

In [30]:
data_dir = "data"
loader = PyPDFLoader(os.path.join(data_dir, "book_chapter_02.pdf"))
docs = loader.load()
len(docs)

34

In [31]:
keywordk_retriever = BM25Retriever.from_documents(docs)

def keyword_document_search(query: str, k: int) -> list[Document]:
    keywordk_retriever.k = k
    return keywordk_retriever.invoke(query)

In [32]:
relevant_keyword_documents = keyword_document_search(
    query="What are morphemes?",
    k=3,
)

print("Keyword search results:")
for i, document in enumerate(relevant_keyword_documents):
    print(f"\n{i+1}. {document.page_content[:10]}")



Keyword search results:

1. 2.6 • R EG

2. 2.1 • W OR

3. 4 CHAPTER 


# Use Cohere Rerank

In [33]:
api_key = os.getenv("COHERE_API_KEY")
co = cohere.Client(api_key)

In [34]:
reranked_hits = co.rerank(
    query="What are morphemes?",
    documents=[doc.page_content for doc in relevant_keyword_documents],
    top_n=10,
    model="rerank-multilingual-v3.0",
)

In [35]:
print("Reranked results:")
for hit in reranked_hits.results:
    print(relevant_keyword_documents[hit.index].page_content[:10])

Reranked results:
4 CHAPTER 
2.1 • W OR
2.6 • R EG
